# Notebook 01 — Carga de Datos y Análisis Exploratorio

**Entrada:** `loan.csv` (Credit Risk Dataset — Kaggle)  
**Salida:** `loan_recodificado.parquet`

## Contenido
1. Configuración del entorno
2. Carga del dataset
3. Exploración inicial de los datos
4. Recodificación de la variable objetivo
5. Análisis exploratorio de datos (EDA)
6. Guardado del dataset recodificado

## 1. Configuración del entorno

In [ ]:
import numpy  as np
import pandas as pd

import matplotlib.pyplot    as plt
import matplotlib.ticker    as mticker
import matplotlib.gridspec  as gridspec
import seaborn              as sns

import warnings
import os

warnings.filterwarnings('ignore')

# Configuracion de visualizacion
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows',    100)
pd.set_option('display.float_format', '{:.4f}'.format)

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 100

PALETA_CLASES = {0: '#2196F3', 1: '#F44336'}   # Azul = bueno, Rojo = malo

In [ ]:
# Rutas del proyecto
DATA_PATH    = os.path.join('..', 'data')
OUTPUT_PATH  = os.path.join('..', 'outputs')
FIGURES_PATH = os.path.join('..', 'figures')

for path in [DATA_PATH, OUTPUT_PATH, FIGURES_PATH]:
    os.makedirs(path, exist_ok=True)

print(f'Rutas configuradas.')
print(f'  data    -> {os.path.abspath(DATA_PATH)}')
print(f'  outputs -> {os.path.abspath(OUTPUT_PATH)}')
print(f'  figures -> {os.path.abspath(FIGURES_PATH)}')

## 2. Carga del dataset

El dataset **Credit Risk Dataset** contiene información de préstamos de LendingClub.
Se puede descargar desde [Kaggle](https://www.kaggle.com/datasets/ranadeep/credit-risk-dataset/data).

> **Nota:** el archivo `loan.csv` debe estar ubicado en `data/loan/loan.csv`.
> Consultar `data/README.md` para instrucciones de descarga.

In [ ]:
ARCHIVO_RAW = os.path.join(DATA_PATH, 'loan', 'loan.csv')

df = pd.read_csv(ARCHIVO_RAW, low_memory=False)

print(f'Dataset cargado correctamente.')
print(f'  Filas     : {df.shape[0]:,}')
print(f'  Columnas  : {df.shape[1]}')
print(f'  Memoria   : {df.memory_usage(deep=True).sum() / 1e6:.1f} MB')

### 2.1 Primer vistazo

In [ ]:
df.head()

### 2.2 Tipos de variables y valores faltantes

In [ ]:
resumen = pd.DataFrame({
    'tipo'         : df.dtypes,
    'no_nulos'     : df.notnull().sum(),
    'nulos'        : df.isnull().sum(),
    'pct_nulos'    : (df.isnull().sum() / len(df) * 100).round(2),
    'n_unicos'     : df.nunique()
})

resumen.sort_values('pct_nulos', ascending=False)

### 2.3 Estadísticas descriptivas — Variables numéricas

In [ ]:
df.describe().T.sort_values('std', ascending=False)

In [ ]:
# Guardado del resumen de variables
ruta_resumen = os.path.join(OUTPUT_PATH, 'resumen_variables.csv')
resumen.sort_values('pct_nulos', ascending=False).to_csv(ruta_resumen, index=True, sep=';')
print(f'Resumen de variables guardado en: {ruta_resumen}')

## 3. Recodificación de la variable objetivo

Se crea la variable binaria `default` a partir de `loan_status` siguiendo las instrucciones del proyecto:

| Categoría original | Codificación | Justificación |
|---|:---:|---|
| `Fully Paid` | **0** | Buen pagador confirmado |
| `Does not meet the credit policy. Status:Fully Paid` | **0** | Buen pagador confirmado |
| `Charged Off` | **1** | Mal pagador confirmado |
| `Late (31-120 days)` | **1** | Se considera mal pagador |
| `Default` | **1** | Impago confirmado |
| `Does not meet the credit policy. Status:Charged Off` | **1** | Mal pagador confirmado |
| `Current` | **NA** | Comportamiento final desconocido |
| `Issued` | **NA** | Sin historial de pago |
| `In Grace Period` | **NA** | Aún no obligado a pagar |
| `Late (16-30 days)` | **NA** | Resultado incierto |

In [ ]:
mapeo_objetivo = {
    'Fully Paid'                                              : 0,
    'Does not meet the credit policy. Status:Fully Paid'     : 0,
    'Charged Off'                                            : 1,
    'Late (31-120 days)'                                     : 1,
    'Default'                                                : 1,
    'Does not meet the credit policy. Status:Charged Off'    : 1,
    'Current'                                                : np.nan,
    'Issued'                                                 : np.nan,
    'In Grace Period'                                        : np.nan,
    'Late (16-30 days)'                                      : np.nan,
}

df['default'] = df['loan_status'].map(mapeo_objetivo)

# Resumen de la recodificacion
n_total  = len(df)
n_buenos = (df['default'] == 0).sum()
n_malos  = (df['default'] == 1).sum()
n_excl   = df['default'].isna().sum()
n_modelo = n_buenos + n_malos

print('Resultado de la recodificacion:')
print(f'  Buenos pagadores (0) : {n_buenos:>8,}  ({n_buenos/n_total*100:.1f}%)')
print(f'  Malos pagadores  (1) : {n_malos:>8,}  ({n_malos/n_total*100:.1f}%)')
print(f'  Excluidos (NA)       : {n_excl:>8,}  ({n_excl/n_total*100:.1f}%)')
print(f'  Total                : {n_total:>8,}')
print(f'\nTasa de incumplimiento (poblacion modelable): {n_malos/n_modelo*100:.1f}%')

## 4. Análisis Exploratorio de Datos (EDA)

A partir de este punto trabajamos con el subconjunto **modelable**, 
es decir, excluyendo los registros donde `default = NA`.

In [ ]:
# Subconjunto modelable (sin NA en la variable objetivo)
df_modelo = df[df['default'].notna()].copy()
df_modelo['default'] = df_modelo['default'].astype(int)

print(f'Subconjunto modelable: {len(df_modelo):,} filas')
print(f'Tasa de incumplimiento: {df_modelo["default"].mean()*100:.1f}%')

### 4.1 Distribución de la variable objetivo

In [ ]:
conteos = df_modelo['default'].value_counts()
pcts    = df_modelo['default'].value_counts(normalize=True).mul(100)
labels  = ['Buen pagador (0)', 'Mal pagador (1)']
colores = [PALETA_CLASES[0], PALETA_CLASES[1]]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Distribucion de la variable objetivo (default)', fontsize=14, fontweight='bold')

# Grafico de barras
bars = axes[0].bar(labels, conteos.values, color=colores, edgecolor='white', linewidth=1.5)
for bar, pct in zip(bars, pcts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + conteos.max()*0.01,
                 f'{bar.get_height():,}\n({pct:.1f}%)',
                 ha='center', va='bottom', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Numero de observaciones')
axes[0].set_title('Conteo absoluto')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
axes[0].set_ylim(0, conteos.max() * 1.15)

# Grafico de torta
wedges, texts, autotexts = axes[1].pie(
    conteos.values, labels=labels, colors=colores,
    autopct='%1.1f%%', startangle=90,
    wedgeprops=dict(edgecolor='white', linewidth=2))
for at in autotexts:
    at.set_fontsize(12)
    at.set_fontweight('bold')
axes[1].set_title('Proporcion relativa')

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_PATH, '01_distribucion_objetivo.png'), bbox_inches='tight')
plt.show()

### 4.2 Variables numéricas — Distribución por clase

In [ ]:
# Variables numericas (excluyendo identificadores y target)
vars_numericas = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
vars_numericas = [v for v in vars_numericas if v not in ['default', 'id', 'member_id']]

n_cols = 4
n_rows = -(-len(vars_numericas) // n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 4))
axes = axes.flatten()

for i, var in enumerate(vars_numericas):
    ax = axes[i]
    for clase, color in PALETA_CLASES.items():
        datos = df_modelo.loc[df_modelo['default'] == clase, var].dropna()
        # Recorte en percentil 99 para limitar el efecto de outliers
        if not datos.empty:
            p99 = datos.quantile(0.99)
            datos = datos[datos <= p99]
        ax.hist(datos, bins=40, alpha=0.6, color=color,
                label=f'default={clase}', density=True, edgecolor='none')

    ax.set_title(var, fontweight='bold', fontsize=11)
    ax.set_xlabel('')
    ax.set_ylabel('Densidad')
    ax.legend(fontsize=9)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Distribucion de variables numericas por clase (default 0 vs 1)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_PATH, '02_distribucion_numericas.png'), bbox_inches='tight')
plt.show()

### 4.3 Variables numéricas — Boxplots por clase

Los boxplots permiten comparar medianas y dispersión entre buenos y malos pagadores.
Se seleccionan automáticamente las 6 variables con mayor diferencia relativa de medianas entre clases.

In [ ]:
# Seleccion automatica de variables con mayor diferencia de medianas entre clases
diferencia_medianas = {}

for var in vars_numericas:
    mediana_0 = df_modelo.loc[df_modelo['default'] == 0, var].median()
    mediana_1 = df_modelo.loc[df_modelo['default'] == 1, var].median()

    if mediana_0 != 0 and pd.notna(mediana_0) and pd.notna(mediana_1):
        diferencia_medianas[var] = abs(mediana_1 - mediana_0) / abs(mediana_0)

# Top 6 variables con mayor diferencia relativa
vars_boxplot = sorted(diferencia_medianas, key=diferencia_medianas.get, reverse=True)[:6]

print('Variables seleccionadas para los boxplots:')
for v in vars_boxplot:
    print(f'  {v}  ->  diferencia relativa de mediana: {diferencia_medianas[v]:.2%}')

n_cols = 3
n_rows = -(-len(vars_boxplot) // n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 5))
axes = axes.flatten()

for i, var in enumerate(vars_boxplot):
    ax = axes[i]
    datos_plot = [
        df_modelo.loc[df_modelo['default'] == 0, var].dropna(),
        df_modelo.loc[df_modelo['default'] == 1, var].dropna()
    ]
    bp = ax.boxplot(datos_plot, patch_artist=True, notch=True, showfliers=False,
                    medianprops=dict(color='black', linewidth=2))

    for patch, color in zip(bp['boxes'], [PALETA_CLASES[0], PALETA_CLASES[1]]):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)

    ax.set_xticks([1, 2])
    ax.set_xticklabels(['Buen pagador (0)', 'Mal pagador (1)'], fontsize=10)
    ax.set_title(var, fontweight='bold', fontsize=11)
    ax.set_ylabel('Valor')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Boxplots de variables numericas por clase',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_PATH, '03_boxplots_numericas.png'), bbox_inches='tight')
plt.show()

### 4.4 Tasa de incumplimiento por variable categórica

In [ ]:
def graficar_tasa_incumplimiento(df, variable, ax, top_n=15):
    """Grafica la tasa de incumplimiento por categoria de una variable."""
    tasa = (
        df.groupby(variable)['default']
          .agg(['mean', 'count'])
          .rename(columns={'mean': 'tasa_incumplimiento', 'count': 'n'})
          .sort_values('tasa_incumplimiento', ascending=True)
          .tail(top_n)
    )

    colores = plt.cm.RdYlGn_r(
        (tasa['tasa_incumplimiento'] - tasa['tasa_incumplimiento'].min()) /
        (tasa['tasa_incumplimiento'].max() - tasa['tasa_incumplimiento'].min() + 1e-9)
    )

    bars = ax.barh(tasa.index.astype(str), tasa['tasa_incumplimiento'] * 100,
                   color=colores, edgecolor='white', linewidth=0.5)

    for bar, (_, row) in zip(bars, tasa.iterrows()):
        ax.text(bar.get_width() + 0.3,
                bar.get_y() + bar.get_height() / 2,
                f"{row['tasa_incumplimiento']*100:.1f}%  (n={row['n']:,})",
                va='center', fontsize=8)

    ax.set_xlabel('Tasa de incumplimiento (%)')
    ax.set_title(variable, fontweight='bold')
    ax.axvline(df['default'].mean() * 100, color='navy', linestyle='--',
               linewidth=1.5, label=f'Media global ({df["default"].mean()*100:.1f}%)')
    ax.legend(fontsize=8)
    ax.set_xlim(0, tasa['tasa_incumplimiento'].max() * 100 * 1.35)

In [ ]:
# Seleccion de variables categoricas
vars_categoricas = df_modelo.select_dtypes(include=['object', 'category']).columns.tolist()

exclude_cols = ['loan_status', 'default', 'issue_d', 'earliest_cr_line',
                'last_pymnt_d', 'next_pymnt_d', 'last_credit_pull_d', 'url', 'desc']
vars_categoricas = [v for v in vars_categoricas if v not in exclude_cols]

print(f'Variables categoricas detectadas ({len(vars_categoricas)}): {vars_categoricas}')

fig, axes = plt.subplots(len(vars_categoricas), 1,
                         figsize=(13, len(vars_categoricas) * 5))
if len(vars_categoricas) == 1:
    axes = [axes]

for ax, var in zip(axes, vars_categoricas):
    graficar_tasa_incumplimiento(df_modelo, var, ax)

fig.suptitle('Tasa de incumplimiento por variable categorica',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_PATH, '04_tasa_incumplimiento_categoricas.png'), bbox_inches='tight')
plt.show()

### 4.5 Mapa de calor de correlaciones

Se visualizan las correlaciones entre las 20 variables numéricas más correlacionadas 
con la variable objetivo, para identificar las más predictivas y detectar posible multicolinealidad.

In [ ]:
# Top 20 variables mas correlacionadas con 'default'
corr_ranking = df_modelo[vars_numericas + ['default']].corr()['default'].abs().sort_values(ascending=False)
top_vars = corr_ranking.head(21).index.tolist()

matriz_corr = df_modelo[top_vars].corr()

fig, ax = plt.subplots(figsize=(14, 11))
mascara = np.triu(np.ones_like(matriz_corr, dtype=bool))

sns.heatmap(
    matriz_corr, mask=mascara, annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.5, cbar_kws={'shrink': 0.8}, ax=ax
)

ax.set_title('Mapa de calor de correlaciones: Top 20 variables con Default',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_PATH, '05_mapa_correlaciones_top.png'), bbox_inches='tight')
plt.show()

### 4.6 Ranking de correlación con `default`

In [ ]:
corr_con_default = (
    df_modelo[top_vars]
    .corr()['default']
    .drop('default')
    .abs()
    .sort_values(ascending=True)
)

# Colores segun signo de la correlacion real
correlaciones_reales = df_modelo[top_vars].corr()['default'].drop('default')[corr_con_default.index]
colores_corr = ['#F44336' if v >= 0 else '#2196F3' for v in correlaciones_reales]

fig, ax = plt.subplots(figsize=(10, 8))
bars = ax.barh(corr_con_default.index, corr_con_default.values,
               color=colores_corr, edgecolor='white')

for bar in bars:
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f'{bar.get_width():.3f}', va='center', fontsize=10, fontweight='bold')

ax.set_xlabel('Correlacion absoluta con Default')
ax.set_title('Top 20 Variables con mayor correlacion con Default', fontweight='bold', fontsize=14)
ax.set_xlim(0, corr_con_default.max() * 1.15)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_PATH, '06_correlacion_con_default_top.png'), bbox_inches='tight')
plt.show()

### 4.7 Análisis de valores faltantes

Antes de pasar al preprocesamiento, es importante visualizar el patrón de datos faltantes
para decidir qué estrategia de imputación utilizar.

In [ ]:
pct_nulos = (df_modelo.isnull().sum() / len(df_modelo) * 100)
pct_nulos = pct_nulos[pct_nulos > 0].sort_values(ascending=True)

if len(pct_nulos) == 0:
    print('No hay valores faltantes en el subconjunto modelable.')
else:
    fig, ax = plt.subplots(figsize=(9, max(5, len(pct_nulos) * 0.4)))

    colores_nulos = ['#FF7043' if p > 20 else '#FFB300' if p > 5 else '#66BB6A'
                     for p in pct_nulos.values]

    bars = ax.barh(pct_nulos.index, pct_nulos.values,
                   color=colores_nulos, edgecolor='white')

    for bar in bars:
        ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
                f'{bar.get_width():.1f}%', va='center', fontsize=9)

    ax.axvline(5,  color='orange', linestyle='--', linewidth=1, alpha=0.7, label='5% umbral moderado')
    ax.axvline(20, color='red',    linestyle='--', linewidth=1, alpha=0.7, label='20% umbral alto')
    ax.set_xlabel('Porcentaje de valores faltantes (%)')
    ax.set_title('Variables con valores faltantes', fontweight='bold')
    ax.legend(fontsize=9)

    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_PATH, '07_valores_faltantes.png'), bbox_inches='tight')
    plt.show()

    print(f'{len(pct_nulos)} variables tienen valores faltantes.')
    print(f'Variables con >20% de faltantes (candidatas a eliminar):')
    print(pct_nulos[pct_nulos > 20].to_string())

## 5. Guardado del dataset recodificado

In [ ]:
RUTA_GUARDADO = os.path.join(DATA_PATH, 'loan_recodificado.parquet')

# Formato Parquet: mas rapido y ligero que CSV
df_modelo.to_parquet(RUTA_GUARDADO, index=False)

size_mb = os.path.getsize(RUTA_GUARDADO) / 1e6
print(f'Dataset guardado: {RUTA_GUARDADO}')
print(f'  Tamano  : {size_mb:.1f} MB')
print(f'  Filas   : {len(df_modelo):,}')
print(f'  Columnas: {df_modelo.shape[1]}')

## Resumen

En este notebook se realizaron las siguientes tareas:

1. **Carga del dataset** desde el archivo CSV descargado de Kaggle.
2. **Exploración inicial**: dimensiones, tipos de datos, valores faltantes y estadísticas descriptivas.
3. **Recodificación de `loan_status`** en una variable binaria `default` (0 = buen pagador, 1 = mal pagador), excluyendo las categorías con resultado incierto.
4. **Análisis exploratorio**:
   - Distribución de la variable objetivo: se observa un desbalance de clases que deberá tratarse en la etapa de preprocesamiento.
   - Histogramas de variables numéricas por clase: variables como `int_rate`, `dti` y `revol_util` muestran distribuciones diferenciadas entre buenos y malos pagadores.
   - Boxplots comparativos: confirman que ciertas variables (como la tasa de interés y los montos de cobro) tienen medianas significativamente distintas entre clases.
   - Tasa de incumplimiento por variable categórica: los grados de riesgo más altos (E, F, G) y ciertos propósitos de préstamo (small business) presentan tasas de incumplimiento superiores a la media.
   - Mapa de correlaciones: identifica variables altamente correlacionadas entre sí (potencial multicolinealidad) y con el target.
   - Valores faltantes: varias variables superan el 40% de nulos, lo que sugiere su eliminación.
5. **Guardado** del dataset recodificado en formato Parquet para la siguiente etapa.

### Hipótesis del EDA

- La **tasa de interés** (`int_rate`) es mayor para los prestatarios que incumplen, lo cual es coherente con el hecho de que los perfiles más riesgosos reciben tasas más altas.
- Los **grados de riesgo** asignados por LendingClub (`grade`, `sub_grade`) son indicadores fuertes de incumplimiento: los grados D–G presentan tasas de impago notablemente superiores.
- El **propósito del préstamo** (`purpose`) influye en el riesgo: préstamos para small business tienen mayor tasa de incumplimiento que los de consolidación de deuda.
- Variables con más del 40% de valores faltantes (como `annual_inc_joint`, `dti_joint`) deben ser eliminadas, ya que cualquier imputación introduciría más ruido que información.
- Existen variables con correlación muy alta entre sí (como `funded_amnt` y `loan_amnt`), lo que indica multicolinealidad que se deberá manejar en la selección de variables.